In [39]:
import re
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import KFold

In [19]:
df = pd.read_excel("data.xlsx")

In [28]:
df.head()

,ID объявления,Тип,Площадь,Возможное назначение,Класс,Метро,Адрес,Здание,Этаж,"Высота потолков, м",Цена,Тип договора,Телефоны,Описание,Парковка,Планировка,Вход,Дополнительно,Ссылка на объявление,Цена за м2
0,329187757,Продажа готового бизнеса,158,Арендный бизнес,NaN,м. Василеостровская (4 мин на машине),"Санкт-Петербург, улица Беринга, 14",NaN,1,NaN,34800000,продажа,+79819346973,Номер объекта: 587727. Продажа готового арендн...,NaN,NaN,NaN,NaN,https://www.cian.ru/sale/commercial/329187757,220253.164557
1,333740002,Продажа помещения свободного назначения,78,"Свободное назначение, Офис, Торговая площадь, ...",NaN,м. Обводный канал (24 мин пешком),"Санкт-Петербург, Витебский Парк ЖК, 6","Жилой дом, Строящееся",1/12,NaN,24887310,NaN,+79110023683,Продаётся нежилое помещение свободного назначе...,NaN,открытая,отдельный с улицы,NaN,https://www.cian.ru/sale/commercial/333740002,319068.076923
2,332051204,Продажа помещения свободного назначения,64,"Свободное назначение, Офис, Торговая площадь, ...",NaN,м. Ладожская (9 мин пешком),"Санкт-Петербург, проспект Энергетиков, 6","Жилой дом, Строящееся",1/15,NaN,26133520,NaN,+79110023683,Продаётся нежилое помещение свободного назначе...,NaN,открытая,отдельный с улицы,NaN,https://www.cian.ru/sale/commercial/332051204,408336.25
3,335392018,Продажа помещения свободного назначения,72,"Свободное назначение, Офис, Торговая площадь, ...",NaN,м. Обводный канал (24 мин пешком),"Санкт-Петербург, Витебский Парк ЖК, 6","Жилой дом, Строящееся",1/12,NaN,25437600,NaN,+79110023683,Продаётся нежилое помещение свободного назначе...,NaN,открытая,отдельный с улицы,NaN,https://www.cian.ru/sale/commercial/335392018,353300.0
4,335391999,Продажа помещения свободного назначения,52,"Свободное назначение, Офис, Торговая площадь, ...",NaN,м. Обводный канал (24 мин пешком),"Санкт-Петербург, Витебский Парк ЖК, 6","Жилой дом, Строящееся",1/12,NaN,17430480,NaN,+79110023683,Продаётся нежилое помещение свободного назначе...,NaN,открытая,отдельный с улицы,NaN,https://www.cian.ru/sale/commercial/335391999,335201.538462


In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID  объявления        200 non-null    int64  
 1   Тип                   200 non-null    str    
 2   Площадь               200 non-null    Int64  
 3   Возможное назначение  195 non-null    str    
 4   Класс                 1 non-null      str    
 5   Метро                 199 non-null    str    
 6   Адрес                 200 non-null    str    
 7   Здание                188 non-null    str    
 8   Этаж                  200 non-null    str    
 9   Высота потолков, м    30 non-null     float64
 10  Цена                  200 non-null    int64  
 11  Тип договора          36 non-null     str    
 12  Телефоны              200 non-null    str    
 13  Описание              200 non-null    str    
 14  Парковка              5 non-null      str    
 15  Планировка            9 non-null  

In [29]:
df.describe()

,ID объявления,Площадь,"Высота потолков, м",Цена,Цена за м2
count,2.000000e+02,200.0,30.000000,2.000000e+02,200.0
mean,3.300214e+08,93.36,3.193667,3.162154e+07,345769.675356
std,4.533557e+06,29.297305,0.480642,1.168005e+07,89411.062603
min,2.983611e+08,50.0,2.500000,1.950000e+06,19500.0
25%,3.300145e+08,73.0,2.835000,2.510749e+07,314377.208336
50%,3.300147e+08,89.0,3.000000,2.997775e+07,348201.857143
75%,3.311867e+08,104.25,3.400000,3.576642e+07,398420.78839
max,3.354315e+08,200.0,4.500000,9.990000e+07,676542.372881


In [32]:
df["Площадь"] = (
    df["Площадь"]
    .astype(str)
    .str.extract(r"(\d+(?:[.,]\d+)?)")[0]
    .str.replace(",", ".", regex=False)
    .astype(float)
    .round() 
    .astype("Int64")
)

In [27]:
df['Цена за м2'] = df['Цена'] / df['Площадь']

In [30]:
345769.675356 * 94

32502349.483464003

In [33]:
df['Метро']

0      м. Василеостровская (4 мин на машине)
1          м. Обводный канал (24 мин пешком)
2                м. Ладожская (9 мин пешком)
3          м. Обводный канал (24 мин пешком)
4          м. Обводный канал (24 мин пешком)
                       ...                  
195      м. Новочеркасская (5 мин на машине)
196      м. Новочеркасская (5 мин на машине)
197      м. Новочеркасская (5 мин на машине)
198      м. Новочеркасская (5 мин на машине)
199      м. Новочеркасская (5 мин на машине)
Name: Метро, Length: 200, dtype: str

In [31]:
df.loc[df['Цена'].idxmin()]

ID  объявления                                                  334555884
Тип                               Продажа торговой площади, Street retail
Площадь                                                               100
Возможное назначение                                                  NaN
Класс                                                                 NaN
Метро                                      м. Ладожская (6 мин на машине)
Адрес                          Санкт-Петербург, Ириновский проспект, 1к11
Здание                                                                NaN
Этаж                                                                  1/1
Высота потолков, м                                                    3.2
Цена                                                              1950000
Тип договора                                                      продажа
Телефоны                                                     +79394075647
Описание                Продаётся 100 

In [40]:
clean_df = df.copy()

# 1. Таргет: цена за квадратный метр
clean_df["price_per_m2"] = clean_df["Цена"].astype(float) / clean_df["Площадь"].astype(float)

# 2. Очистка метро через безопасные строковые методы pandas (без лямбд и без ошибок типов)
metro_series = clean_df["Метро"].fillna("Не указано").astype(str)

clean_df["metro_clean"] = (
    metro_series
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.replace("м.", "", regex=False)
    .str.strip()
)

# Время до метро (минуты)
clean_df["metro_time"] = (
    metro_series
    .str.extract(r"\((\d+)\s*мин")[0]
    .astype(float)
)
clean_df["metro_time"] = clean_df["metro_time"].fillna(clean_df["metro_time"].median())

# 3. Категория этажа
def parse_floor(val):
    if pd.isna(val):
        return "прочее"
    val_str = str(val).lower()
    if any(w in val_str for w in ["цоколь", "подвал", "-1"]):
        return "подвал/цоколь"
    match = re.search(r"\d+", val_str)
    if match:
        return "1 этаж" if int(match.group(0)) == 1 else "2+ этаж"
    return "прочее"

clean_df["floor_group"] = clean_df["Этаж"].apply(parse_floor)

# Входная группа
clean_df["entrance_clean"] = clean_df["Вход"].fillna("не указан").astype(str).str.strip()

# 4. Мое помещение
my_property = pd.DataFrame([{
    "Площадь": 94,
    "metro_clean": "Рыбацкое",
    "metro_time": 10.0,
    "floor_group": "1 этаж",       # '1 этаж', 'подвал/цоколь', '2+ этаж'
    "entrance_clean": "с улицы"    # 'с улицы', 'со двора', 'не указан'
}])

feature_cols = ["Площадь", "metro_clean", "metro_time", "floor_group", "entrance_clean"]
train_base = clean_df.dropna(subset=feature_cols + ["price_per_m2"]).copy()

# Приведение типов перед кодированием
train_base["Площадь"] = train_base["Площадь"].astype(float)
my_property["Площадь"] = my_property["Площадь"].astype(float)

# One-Hot Encoding
combined = pd.concat([train_base[feature_cols], my_property], ignore_index=True)
combined_encoded = pd.get_dummies(
    combined, 
    columns=["metro_clean", "floor_group", "entrance_clean"], 
    drop_first=False
)

X = combined_encoded.iloc[:-1].astype(float)
y = train_base["price_per_m2"].astype(float)
X_target = combined_encoded.iloc[[-1]].astype(float)

# 5. Оценка качества модели (K-Fold кросс-валидация)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mape_scores = []

for train_idx, val_idx in kf.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = RandomForestRegressor(n_estimators=150, max_depth=8, random_state=42)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)
    mape_scores.append(mean_absolute_percentage_error(y_val, preds) * 100)

print(f"Средняя ошибка модели (MAPE): {np.mean(mape_scores):.1f}%")

# 6. Финальный расчет
final_rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
final_rf.fit(X, y)

pred_m2 = final_rf.predict(X_target)[0]
total_pred = pred_m2 * my_property["Площадь"].values[0]

print("\n--- РЕЗУЛЬТАТ ОЦЕНКИ ---")
print(f"Расчетная цена за м²: {pred_m2:,.0f} руб.")
print(f"Рыночная стоимость помещения: {total_pred:,.0f} руб.")
print(f"Диапазон цен предложения: {total_pred * 0.95:,.0f} — {total_pred * 1.05:,.0f} руб.")
print(f"С учетом торга (дисконт ~8%): {total_pred * 0.92:,.0f} руб.")

Средняя ошибка модели (MAPE): 17.9%

--- РЕЗУЛЬТАТ ОЦЕНКИ ---
Расчетная цена за м²: 371,609 руб.
Рыночная стоимость помещения: 34,931,279 руб.
Диапазон цен предложения: 33,184,715 — 36,677,843 руб.
С учетом торга (дисконт ~8%): 32,136,776 руб.


## Анализ точности модели и итоговый отчет об оценке помещения

### 1. Оценка качества модели и специфика выборки
* **Метрика качества (MAPE = 17.9%):** Средняя абсолютная процентная ошибка на кросс-валидации составила **17.9%**. В сегменте помещений свободного назначения (ПСН) для выборки в 200 объектов такой показатель свидетельствует об устойчивости модели и отсутствии критического переобучения.
* **Особенность локации (м. Рыбацкое):** В обучающей выборке объекты у данной станции метро либо отсутствуют. В связи с этим алгоритм применил базовую оценку для объектов аналогичного формата (площадь ~90–100 м², 1 этаж, отдельный вход с улицы), усредненную по спальным районам выборки.
* **Факторы неполноты данных:** Модель опирается на формальные атрибуты объявления и не учитывает скрытые факторы ценности (пешеходный трафик, электрическая мощность, наличие витрин, статус действующего договора аренды с МАП).

---

### 2. Ценовые ориентиры объекта (ПСН, 94 кв. м, Рыбацкое, 1 этаж, вход с улицы)

| Показатель | Значение | Пояснение к расчету |
| :--- | :--- | :--- |
| **Удельная ставка за м²** | **371 600 руб./м²** | Индикативная расчетная стоимость квадратного метра для формата стрит-ритейл |
| **Рыночная стоимость помещения** | **34 930 000 руб.** | Математическое ожидание стоимости при текущей конфигурации параметров |
| **Коридор цен экспозиции** | **33 180 000 — 36 680 000 руб.** | Диапазон цен (±5%) для старта открытых продаж на агрегаторах |
| **Ожидаемая цена сделки** | **~32 140 000 руб.** | Расчетная сумма закрытия сделки с учетом дисконта на торг (~8%) |

---

### 3. Итоговые выводы и прикладные рекомендации

1. **Стратегия позиционирования на рынке:**
   * Помещение площадью 94 м² на 1 этаже со входом с улицы в спальном массиве относится к наиболее востребованной категории коммерческой недвижимости под ритейл шаговой доступности (ПВЗ, аптеки, алкомаркеты, пекарни, лаборатории).
   * Рекомендуется начинать экспозицию в диапазоне **35.0 — 36.5 млн руб.**, закладывая стандартный переговорный запас для предоставления скидки покупателю до уровня **32 — 33 млн руб.**
2. **Корректировка цены на основе физических характеристик:**
   * **Повышающие факторы (сдвиг к 36+ млн руб.):** расположение на первой линии вдоль основного пешеходного потока от жилых массивов к метро/остановкам, витринное остекление в пол, выделенная электрическая мощность от 15–20 кВт, отсутствие ступеней на входе.
   * **Понижающие факторы (сдвиг к 30–32 млн руб.):** расположение во внутреннем дворе/стилобате без сквозной видимости с улицы, сложная геометрия (коридорная система, несущие колонны), ограничение по мощности (менее 10 кВт), жесткая конкуренция с коммерческими площадями в соседних новостройках.